Agent Environment Loop

In [96]:
import abc, time, random, queue, threading, numpy as np
from typing import Any, Dict, List, Tuple, Optional
from dataclasses import dataclass, field
import gymnasium as gym

Base Agent Class Interface

In [97]:
class Agent(abc.ABC):
    """Abstract Base Class defining standard RL Agent Interface"""
    
    @abc.abstractmethod
    def observe(self, observation: Any) -> None:
        """Receive and process the current environment state or observation"""
        pass

    @abc.abstractmethod
    def act(self) -> int:
        """Select and return an action based on the agent's current policy"""
        pass

    @abc.abstractmethod
    def update(self, reward: float, terminated: bool, truncated: bool) -> None:
        """Update internal weights, tables or history based on environment feedback"""
        pass

Initialise Random Agent

In [98]:
class RandomAgent(Agent):
    def __init__(self, action_space: gym.spaces.Discrete):
        self.action_space = action_space
    
    # A random agent does not observe neither does it update its internal state. It acts randomly.
    def observe(self, observation: Any) -> None:
        pass

    def update(self, reward: float, terminated: bool, truncated: bool) -> None:
        pass

    def act(self) -> int:
        return int(self.action_space.sample())

Initializing Cartpole Heuristics Agent

In [99]:
class CartpoleHeuristicsAgent(Agent):
    """Move the cart in the same direction the pole is tilting to balance it... as simple as that"""
    def __init__(self) -> None:
        self.pole_angle = 0.0
    
    def observe(self, observation: Any) -> None:
        # observation layout: [cart_pos, cart_vel, pole_angle, pole_vel]
        self.pole_angle = float(observation[2])
    
    # This agent uses fixed logic and does not learn
    def update(self, reward: float, terminated: bool, truncated: bool) -> None:
        pass

    def act(self) -> int:
        return 1 if self.pole_angle > 0 else 0

Initializing MiniGrid Heuristics Agent

In [100]:
class MiniGridHeuristicsAgent(Agent):
    """Rule: Move forward, if block by a wall, rotate right"""
    def __init__(self) -> None:
        self.is_front_clear: bool = True

    def observe(self, observation: Any) -> None:
        # observation['image'] is (x, y, channel); agent sits at (3, 6) facing
        # decreasing y, so the cell directly ahead is (3, 5), not (5, 3).
        grid = observation['image']
        front_cell = grid[3, 5]
        self.is_front_clear = (front_cell[0] != 2)
    
    def update(self, reward: float, terminated: bool, truncated: bool) -> None:
        pass

    def act(self) -> int:
        # 2 - forward, 1 - right, 0 - left
        return 2 if self.is_front_clear else 1

Initializing TableBased Agent

In [101]:
class TableBasedAgent(Agent):
    """A simple tabular Q-learning agent"""
    def __init__(self, action_space: gym.spaces.Discrete, is_vector_env: bool = True):
        self.action_space = action_space
        self.is_vector_env = is_vector_env
        self.q_table: Dict[Tuple[int, ...], Dict[int, float]] = {}
        self.current_state_key: Optional[Tuple[int, ...]] = None
        self.last_state_key: Optional[Tuple[int, ...]] = None
        self.last_action: Optional[int] = None

        # Hyperparameters
        self.alpha = 0.1 # the learning rate
        self.gamma = 0.99 # the discount factor - does it prioritise long-term rewards over immediate satisfaction
        self.epsilon = 0.1 # the exploration rate - vs exploitation
    
    def _discretize(self, observation: Any) -> Tuple[int, ...]:
        """Maps complex observation layouts to discrete hashable tuples"""
        if self.is_vector_env: # for cartpole
            return tuple(np.round(observation, decimals=1).tolist())
        else: # for minigrid
            pos_encoding = tuple(observation['image'][:, :, 0].flatten().tolist())
            return (observation['direction'], *pos_encoding)
    
    def observe(self, observation: Any) -> None:
        state_key = self._discretize(observation)
        self.last_state_key = self.current_state_key
        self.current_state_key = state_key
        if state_key not in self.q_table:
            self.q_table[state_key] = {a: 0.0 for a in range(self.action_space.n)}
    
    def act(self) -> int:
        state_values = self.q_table[self.current_state_key]
        if random.random() < self.epsilon:
            self.last_action = int(self.action_space.sample())
        else:
            self.last_action = max(state_values, key=state_values.get)
        return self.last_action
    
    def update(self, reward: float, terminated: bool, truncated: bool) -> None:
        if self.last_state_key is None or self.last_action is None:
            return
        current_q = self.q_table[self.last_state_key][self.last_action]
        max_future_q = 0.0 if (terminated or truncated) else max(self.q_table[self.current_state_key].values())
        self.q_table[self.last_state_key][self.last_action] = current_q + self.alpha * (
            reward + self.gamma * max_future_q - current_q
        )

Initiatlize DelayedMalfunctioning Agent

In [102]:
class DelayedMalfunctioningAgent(Agent):
    """Slow agent to test safety budgets"""
    def observe(self, observation: Any) -> None:
        pass
    def update(self, reward: float, terminated: bool, truncated: bool) -> None:
        pass
    def act(self) -> int:
        time.sleep(0.05) # forcing a 50ms delay
        return 0

Logging Infra and Budget Runner

In [103]:
@dataclass
class EpisodicLog:
    agent_name: str
    environment_name: str
    episodic_return: float = 0.0
    episode_length: int = 0
    action_distribution: Dict[int, int] = field(default_factory=dict)
    failure_states: List[int] = field(default_factory=list)

In [104]:
def _worker_act(agent: Agent, result_queue: "queue.Queue"):
    try:
        action = agent.act()
        result_queue.put((True, action))
    except Exception as e:
        result_queue.put((False, str(e)))

In [105]:
def get_action_with_budget(agent: Agent, time_budget: float) -> Tuple[Optional[int], Optional[str]]:
    result_queue: "queue.Queue" = queue.Queue()
    worker = threading.Thread(target=_worker_act, args=(agent, result_queue), daemon=True)
    worker.start()
    worker.join(timeout=time_budget)

    if worker.is_alive():
        return None, f"Timeout budget exceeded (> {time_budget*1000:.1f}ms)"

    if not result_queue.empty():
        success, val = result_queue.get()
        if success:
            return val, None
        return None, f"Execution Error: {val}"
    return None, "Worker thread finished without producing a result"

Run the RL loop

In [106]:
def run_online_loop(agent: Agent, env_id: str, agent_name: str, max_episodes: int = 5, time_budget: float = 0.01) -> List[EpisodicLog]:
    env = gym.make(env_id)
    history: List[EpisodicLog] = []

    for ep in range(max_episodes):
        log = EpisodicLog(agent_name=agent_name, environment_name=env_id)
        observation, info = env.reset()
        terminated, truncated = False, False

        while not (truncated or terminated):
            log.episode_length += 1
            agent.observe(observation)
            action, failure = get_action_with_budget(agent, time_budget)

            if failure:
                log.failure_states.append(f"Step {log.episode_length} Failed: {failure}")
                break
            log.action_distribution[action] = log.action_distribution.get(action, 0) + 1
            observation, reward, terminated, truncated, info = env.step(action)
            agent.update(reward, terminated, truncated)
            log.episodic_return += reward

        history.append(log)
    env.close()
    return history

In [107]:
if __name__ == "__main__":
    LATENCY_BUDGET = 0.010  # Strict 10ms execution limit per step
    EPISODES_PER_RUN = 3

    evaluation_matrix = [
        {"env": "CartPole-v1", "name": "Random_Agent", "agent": RandomAgent(gym.make("CartPole-v1").action_space)},
        {"env": "CartPole-v1", "name": "CartPole_Heuristic", "agent": CartpoleHeuristicsAgent()},
        {"env": "CartPole-v1", "name": "Tabular_Q_Learning", "agent": TableBasedAgent(gym.make("CartPole-v1").action_space, is_vector_env=True)},
        {"env": "CartPole-v1", "name": "Defective_Slow_Agent", "agent": DelayedMalfunctioningAgent()},
        {"env": "MiniGrid-Empty-5x5-v0", "name": "Random_Agent", "agent": RandomAgent(gym.make("MiniGrid-Empty-5x5-v0").action_space)},
        {"env": "MiniGrid-Empty-5x5-v0", "name": "MiniGrid_Heuristic", "agent": MiniGridHeuristicsAgent()},
        {"env": "MiniGrid-Empty-5x5-v0", "name": "Tabular_Q_Learning", "agent": TableBasedAgent(gym.make("MiniGrid-Empty-5x5-v0").action_space, is_vector_env=False)}
    ]
    print(f"=== Starting WM0.1 Agent Evaluation Loop (Budget: {LATENCY_BUDGET*1000:.1f}ms) ===\n")
    
    for task in evaluation_matrix:
        print(f"Running: Evaluation of {task['name']} on {task['env']}...")
        results = run_online_loop(
            agent=task["agent"],
            env_id=task["env"],
            agent_name=task["name"],
            max_episodes=EPISODES_PER_RUN,
            time_budget=LATENCY_BUDGET
        )
        
        # Aggregate and present the tracking metrics
        for idx, log in enumerate(results):
            print(f"Episode {idx + 1}:")
            print(f"Return: {log.episodic_return:.2f} | Length: {log.episode_length}") 
            print(f"Action Mix: {dict(log.action_distribution)}")
            if log.failure_states:
                print(f"[FAILURES ENCOUNTERED]: {log.failure_states}")
        
        print("-" * 70)

=== Starting WM0.1 Agent Evaluation Loop (Budget: 10.0ms) ===

Running: Evaluation of Random_Agent on CartPole-v1...
Episode 1:
Return: 19.00 | Length: 19
Action Mix: {1: 8, 0: 11}
Episode 2:
Return: 11.00 | Length: 11
Action Mix: {1: 9, 0: 2}
Episode 3:
Return: 17.00 | Length: 17
Action Mix: {1: 6, 0: 11}
----------------------------------------------------------------------
Running: Evaluation of CartPole_Heuristic on CartPole-v1...
Episode 1:
Return: 31.00 | Length: 31
Action Mix: {1: 16, 0: 15}
Episode 2:
Return: 47.00 | Length: 47
Action Mix: {0: 25, 1: 22}
Episode 3:
Return: 44.00 | Length: 44
Action Mix: {1: 20, 0: 24}
----------------------------------------------------------------------
Running: Evaluation of Tabular_Q_Learning on CartPole-v1...
Episode 1:
Return: 13.00 | Length: 13
Action Mix: {1: 2, 0: 11}
Episode 2:
Return: 12.00 | Length: 12
Action Mix: {0: 11, 1: 1}
Episode 3:
Return: 9.00 | Length: 9
Action Mix: {0: 9}
----------------------------------------------------

Episode 1:
Return: 0.00 | Length: 1
Action Mix: {}
[FAILURES ENCOUNTERED]: ['Step 1 Failed: Timeout budget exceeded (> 10.0ms)']
Episode 2:
Return: 0.00 | Length: 1
Action Mix: {}
[FAILURES ENCOUNTERED]: ['Step 1 Failed: Timeout budget exceeded (> 10.0ms)']
Episode 3:
Return: 0.00 | Length: 1
Action Mix: {}
[FAILURES ENCOUNTERED]: ['Step 1 Failed: Timeout budget exceeded (> 10.0ms)']
----------------------------------------------------------------------
Running: Evaluation of Random_Agent on MiniGrid-Empty-5x5-v0...
Episode 1:
Return: 0.10 | Length: 100
Action Mix: {3: 13, 5: 12, 0: 21, 6: 18, 4: 12, 2: 15, 1: 9}
Episode 2:
Return: 0.00 | Length: 100
Action Mix: {6: 18, 4: 23, 1: 8, 2: 12, 5: 10, 0: 11, 3: 18}
Episode 3:
Return: 0.00 | Length: 100
Action Mix: {0: 16, 1: 17, 6: 16, 5: 9, 2: 17, 3: 12, 4: 13}
----------------------------------------------------------------------
Running: Evaluation of MiniGrid_Heuristic on MiniGrid-Empty-5x5-v0...
Episode 1:
Return: 0.95 | Length: 5
Acti